In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!git clone https://github.com/yzygit1230/DFLNet.git
%cd DFLNet

Cloning into 'DFLNet'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 38 (delta 7), reused 24 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 1.77 MiB | 12.00 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/kaggle/working/DFLNet


In [2]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
print(os.listdir("/kaggle/input/datasets"))

['viswapathiharshini']


In [4]:
print(os.listdir("/kaggle/input/datasets/viswapathiharshini"))

['busi-whu']


In [5]:
import shutil
import os

src_dataset = "/kaggle/input/datasets/viswapathiharshini/busi-whu"
dst_dataset = "datasets/BUSI-WHU"

if not os.path.exists(dst_dataset):
    shutil.copytree(src_dataset, dst_dataset)

print("Dataset copied successfully!")

Dataset copied successfully!


In [6]:
for root, dirs, files in os.walk("datasets/BUSI-WHU"):
    print(root, len(files))

datasets/BUSI-WHU 0
datasets/BUSI-WHU/valid 0
datasets/BUSI-WHU/valid/gt 186
datasets/BUSI-WHU/valid/img 186
datasets/BUSI-WHU/test 0
datasets/BUSI-WHU/test/gt 186
datasets/BUSI-WHU/test/img 186
datasets/BUSI-WHU/train 0
datasets/BUSI-WHU/train/gt 0
datasets/BUSI-WHU/train/gt/ori 555
datasets/BUSI-WHU/train/img 0
datasets/BUSI-WHU/train/img/ori 555


In [7]:
import shutil
import os

base = "datasets/BUSI-WHU/train"

# move images
src_img = os.path.join(base, "img/ori")
dst_img = os.path.join(base, "img")

for f in os.listdir(src_img):
    shutil.move(os.path.join(src_img, f), dst_img)

# move masks
src_gt = os.path.join(base, "gt/ori")
dst_gt = os.path.join(base, "gt")

for f in os.listdir(src_gt):
    shutil.move(os.path.join(src_gt, f), dst_gt)

print("Train images and masks moved!")

Train images and masks moved!


In [8]:
import shutil

shutil.rmtree("datasets/BUSI-WHU/train/img/ori")
shutil.rmtree("datasets/BUSI-WHU/train/gt/ori")

print("ori folders removed")

ori folders removed


In [9]:
import os

base = "datasets/BUSI-WHU"

for split in ["train","valid","test"]:
    gt_path = f"{base}/{split}/gt"
    for file in os.listdir(gt_path):
        if "_anno.bmp" in file:
            new_name = file.replace("_anno", "")
            os.rename(os.path.join(gt_path, file),
                      os.path.join(gt_path, new_name))

print("Masks renamed successfully!")

Masks renamed successfully!


In [10]:
!pip install Dropblock

In [11]:
!ls

dataset.py  eval.py  models	train.py  visualization.py
datasets    Fig      README.md	util


In [12]:
%%writefile train.py

import torch
from util.helpers import get_criterion
import os
import pandas as pd
from tqdm import tqdm
import random
import numpy as np
from torch.utils.data import DataLoader
from models.Base import dropblock_step
from util.common import check_dirs, CosOneCycle, ScaleInOutput
import argparse
from models.DFLNet import DFLNet
from util.transforms import train_transforms, test_transforms
from glob import glob
from dataset import BreastData

# ---------------- Arguments ----------------

parser = argparse.ArgumentParser('DFLNet Train')

parser.add_argument("--network", type=str, default="DFLNet")
parser.add_argument("--inplanes", type=int, default=64)
parser.add_argument("--loss_function", type=str, default="hybrid")
parser.add_argument("--input_size", type=int, default=256)
parser.add_argument("--num_workers", type=int, default=2)
parser.add_argument("--batch_size", type=int, default=16)
parser.add_argument("--learning_rate", type=float, default=0.00035)
parser.add_argument("--epochs", type=int, default=400) 
parser.add_argument("--pretrain_pth", type=str, default='None')

# Resume arguments
parser.add_argument("--resume", type=str, default='None',
                    help="Path to checkpoint")
parser.add_argument("--start_epoch", type=int, default=0)

opt = parser.parse_args()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ---------------- Seed ----------------

def seed_torch(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

seed_torch()

# ---------------- Dataset ----------------

train_path = "./datasets/BUSI-WHU/train/"
val_path = "./datasets/BUSI-WHU/valid/"

train_data = pd.DataFrame({
    'images': sorted(glob(train_path + "img/*.bmp")),
    'masks': sorted(glob(train_path + "gt/*.bmp"))
})

val_data = pd.DataFrame({
    'images': sorted(glob(val_path + "img/*.bmp")),
    'masks': sorted(glob(val_path + "gt/*.bmp"))
})

train_dataset = BreastData(train_data, train_transforms)
val_dataset = BreastData(val_data, test_transforms)

train_loader = DataLoader(train_dataset,
                          batch_size=opt.batch_size,
                          shuffle=True,
                          num_workers=opt.num_workers)

val_loader = DataLoader(val_dataset,
                        batch_size=1,
                        shuffle=False,
                        num_workers=opt.num_workers)

# ---------------- Model ----------------

save_path = check_dirs()

model = DFLNet(opt).to(device)

criterion = get_criterion(opt.loss_function)

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=opt.learning_rate,
                              weight_decay=0.0001)

scheduler = CosOneCycle(optimizer,
                        max_lr=opt.learning_rate,
                        epochs=opt.epochs,
                        up_rate=0)

scale = ScaleInOutput(opt.input_size)

# ---------------- Resume ----------------

start_epoch = opt.start_epoch
best_val_f1 = 0.0

if opt.resume != 'None' and os.path.exists(opt.resume):

    print(f"Resuming from {opt.resume}")

    checkpoint = torch.load(opt.resume, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch']
    best_val_f1 = checkpoint.get('val_f1', 0.0)

    print(f"Resumed from epoch {start_epoch}")

# Paths
last_ckpt_path = os.path.join(save_path, "last_checkpoint.pt")
best_ckpt_path = os.path.join(save_path, "best_checkpoint.pt")

print("\nEpoch | Train Loss | Val Loss | Train Acc | Val Acc | Val F1")

# ---------------- Training ----------------

for epoch in range(start_epoch, opt.epochs):

    model.train()

    train_loss = 0
    correct_pixels = 0
    total_pixels = 0

    for batch_img, labels, _ in tqdm(train_loader):

        batch_img = batch_img.float().to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()

        batch_img, _ = scale.scale_input((batch_img, batch_img))

        outputs = model(batch_img)
        outputs = scale.scale_output(outputs)

        loss = criterion(outputs, labels, device)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        pred_out = outputs[-1]
        _, preds = torch.max(pred_out, 1)

        correct_pixels += (preds == labels).sum().item()
        total_pixels += labels.numel()

    train_acc = correct_pixels / total_pixels
    train_loss /= len(train_loader)

    scheduler.step()
    dropblock_step(model)

    # ---------- Validation ----------

    model.eval()

    val_loss = 0
    val_tp = val_fp = val_fn = val_tn = 0
    eps = 1e-8

    with torch.no_grad():

        for batch_img, labels, _ in val_loader:

            batch_img = batch_img.float().to(device)
            labels = labels.long().to(device)

            batch_img, _ = scale.scale_input((batch_img, batch_img))

            outputs = model(batch_img)
            outputs = scale.scale_output(outputs)

            loss = criterion(outputs, labels, device)
            val_loss += loss.item()

            pred_out = outputs[-1]
            _, preds = torch.max(pred_out, 1)

            labels_np = labels.cpu().numpy()
            preds_np = preds.cpu().numpy()

            val_tp += np.sum((labels_np == 1) & (preds_np == 1))
            val_fp += np.sum((labels_np == 0) & (preds_np == 1))
            val_fn += np.sum((labels_np == 1) & (preds_np == 0))
            val_tn += np.sum((labels_np == 0) & (preds_np == 0))

    val_loss /= len(val_loader)

    val_acc = (val_tp + val_tn) / (val_tp + val_tn + val_fp + val_fn + eps)
    val_precision = val_tp / (val_tp + val_fp + eps)
    val_recall = val_tp / (val_tp + val_fn + eps)
    val_f1 = 2 * val_precision * val_recall / (val_precision + val_recall + eps)

    print(f"{epoch+1:3d} | {train_loss:.4f} | {val_loss:.4f} | "
          f"{train_acc*100:.2f}% | {val_acc*100:.2f}% | {val_f1*100:.2f}%")

    # 🔹 ALWAYS SAVE LAST CHECKPOINT
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_f1': val_f1,
    }, last_ckpt_path)

    # 🔹 SAVE BEST MODEL
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
        }, best_ckpt_path)

        print(f"  --> Saved BEST model (F1: {val_f1*100:.2f}%)")

    # 🔹 OPTIONAL: SAVE EVERY 10 EPOCHS
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(),
                   os.path.join(save_path, f"model_epoch_{epoch+1}.pth"))



Overwriting train.py


In [13]:
%%writefile eval.py
import torch
from models.DFLNet import DFLNet
from statistics import mean
from tqdm import tqdm
from util.AverageMeter import compute_assd
from util.transforms import test_transforms
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
from glob import glob
from dataset import BreastData
import os
import argparse

# ---------------- Arguments ----------------
parser = argparse.ArgumentParser()
parser.add_argument("--ckpt", type=str, default=None,
                    help="Path to checkpoint (optional)")
args = parser.parse_args()

# ---------------- Device ----------------
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ---------------- Dataset ----------------
test_path = "./datasets/BUSI-WHU/test/"

test_data = pd.DataFrame({
    'images': sorted(glob(test_path + "img/*.bmp")),
    'masks': sorted(glob(test_path + "gt/*.bmp"))
})

if len(test_data) == 0:
    raise FileNotFoundError(f"No test images found at {test_path}")

print(f"Test set size: {len(test_data)} images")

test_dataset = BreastData(test_data, transforms=test_transforms)

test_loader = DataLoader(test_dataset,
                         batch_size=1,
                         shuffle=False,
                         num_workers=2)

# ---------------- Load Model ----------------

if args.ckpt is not None:
    path = args.ckpt
else:
    train_folders = sorted(glob("runs/train/*"))

    if not train_folders:
        raise FileNotFoundError("No training folders found")

    latest_folder = train_folders[-1]
    path = os.path.join(latest_folder, "best_checkpoint.pt")

    if not os.path.exists(path):
        raise FileNotFoundError(f"No best_checkpoint.pt found in {latest_folder}")

print("Loading model:", path)

checkpoint = torch.load(path, map_location=device, weights_only=False)

# Dummy opt
class Opt:
    def __init__(self):
        self.inplanes = 64
        self.network = "DFLNet"
        self.input_size = 256
        self.pretrain_pth = 'None'

opt = Opt()

model = DFLNet(opt).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ---------------- Evaluation ----------------

THRESHOLD = 0.35   # 🔥 try 0.3 / 0.35 / 0.4

tn = fp = fn = tp = 0
assd_values = []

with torch.no_grad():

    for batch_img, labels, _ in tqdm(test_loader):

        batch_img = batch_img.float().to(device)
        labels = labels.long().to(device)

        # -------- TTA START --------

        # original
        out1 = model(batch_img)

        # flipped
        img_flip = torch.flip(batch_img, dims=[-1])
        out2 = model(img_flip)

        # handle multi-output
        if isinstance(out1, (list, tuple)):
            out1 = out1[-1]
        if isinstance(out2, (list, tuple)):
            out2 = out2[-1]

        # flip back
        out2 = torch.flip(out2, dims=[-1])

        # average
        final_output = (out1 + out2) / 2

        # -------- TTA END --------

        # -------- THRESHOLDING --------
        probs = torch.softmax(final_output, dim=1)[:, 1, :, :]
        preds = (probs > THRESHOLD).long()
        # --------------------------------

        labels_np = labels.cpu().numpy()
        preds_np = preds.cpu().numpy()

        assd_values.append(compute_assd(preds_np, labels_np))

        tp += np.sum((labels_np == 1) & (preds_np == 1))
        tn += np.sum((labels_np == 0) & (preds_np == 0))
        fp += np.sum((labels_np == 0) & (preds_np == 1))
        fn += np.sum((labels_np == 1) & (preds_np == 0))

# ---------------- Metrics ----------------

eps = 1e-8

precision = tp / (tp + fp + eps)
recall = tp / (tp + fn + eps)
f1 = 2 * precision * recall / (precision + recall + eps)

iou0 = tn / (tn + fp + fn + eps)
iou1 = tp / (tp + fp + fn + eps)
miou = (iou0 + iou1) / 2

oa = (tp + tn) / (tp + tn + fp + fn + eps)

total = tp + tn + fp + fn
po = (tp + tn) / total
pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total * total)

kappa = (po - pe) / (1 - pe + eps)

# ---------------- Output ----------------

print("\nConfusion Matrix")
print(f"TP: {tp}  FP: {fp}")
print(f"FN: {fn}  TN: {tn}")

print("\nFINAL TEST RESULTS")
print(f"Precision:        {round(precision * 100, 2)} %")
print(f"Recall:           {round(recall * 100, 2)} %")
print(f"F1 Score:         {round(f1 * 100, 2)} %")
print(f"Mean IoU:         {round(miou * 100, 2)} %")
print(f"Overall Accuracy: {round(oa * 100, 2)} %")
print(f"Kappa:            {round(kappa * 100, 2)} %")
print(f"ASSD:             {round(mean(assd_values), 4)}")

Overwriting eval.py


In [14]:
!python train.py

INFO:numexpr.utils:NumExpr defaulting to 4 threads.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

------------------------------Check Dirs------------------------------
checkpoints & results are saved at: ./runs/train/1

Epoch | Train Loss | Val Loss | Train Acc | Val Acc | Val F1
  0%|                                                    | 0/35 [00:00<?, ?it/s]/kaggle/working/DFLNet/util/metrics.py:44: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  logpt = F.log_softmax(input)
/kaggle/working/DFLNet/models/reverse_function.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  torch.cuda.amp.autocast(**ctx.

In [15]:
!python eval.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Test set size: 186 images
Loading model: runs/train/1/best_checkpoint.pt
100%|█████████████████████████████████████████| 186/186 [00:13<00:00, 14.02it/s]

Confusion Matrix
TP: 773989  FP: 73926
FN: 110183  TN: 11231598

FINAL TEST RESULTS
Precision:        91.28 %
Recall:           87.54 %
F1 Score:         89.37 %
Mean IoU:         89.59 %
Overall Accuracy: 98.49 %
Kappa:            88.56 %
ASSD:             0.899


In [18]:
%%writefile eval.py
import torch
from models.DFLNet import DFLNet
from statistics import mean
from tqdm import tqdm
from util.AverageMeter import compute_assd
from util.transforms import test_transforms
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
from glob import glob
from dataset import BreastData
import os
import argparse

# ---------------- Arguments ----------------
parser = argparse.ArgumentParser()
parser.add_argument("--ckpt", type=str, default=None,
                    help="Path to checkpoint (optional)")
args = parser.parse_args()

# ---------------- Device ----------------
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ---------------- Dataset ----------------
test_path = "./datasets/BUSI-WHU/test/"

test_data = pd.DataFrame({
    'images': sorted(glob(test_path + "img/*.bmp")),
    'masks': sorted(glob(test_path + "gt/*.bmp"))
})

if len(test_data) == 0:
    raise FileNotFoundError(f"No test images found at {test_path}")

print(f"Test set size: {len(test_data)} images")

test_dataset = BreastData(test_data, transforms=test_transforms)

test_loader = DataLoader(test_dataset,
                         batch_size=1,
                         shuffle=False,
                         num_workers=2)

# ---------------- Load Model ----------------
if args.ckpt is not None:
    path = args.ckpt
else:
    train_folders = sorted(glob("runs/train/*"))

    if not train_folders:
        raise FileNotFoundError("No training folders found")

    latest_folder = train_folders[-1]
    path = os.path.join(latest_folder, "best_checkpoint.pt")

    if not os.path.exists(path):
        raise FileNotFoundError(f"No best_checkpoint.pt found in {latest_folder}")

print("Loading model:", path)

checkpoint = torch.load(path, map_location=device, weights_only=False)

# Dummy opt
class Opt:
    def __init__(self):
        self.inplanes = 64
        self.network = "DFLNet"
        self.input_size = 256
        self.pretrain_pth = 'None'

opt = Opt()

model = DFLNet(opt).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ---------------- Evaluation ----------------

THRESHOLD = 0.35   # 🔥 try 0.3 / 0.32 / 0.38 / 0.4

tn = fp = fn = tp = 0
assd_values = []

with torch.no_grad():

    for batch_img, labels, _ in tqdm(test_loader):

        batch_img = batch_img.float().to(device)
        labels = labels.long().to(device)

        # -------- STRONG TTA START --------

        # original
        out1 = model(batch_img)

        # horizontal flip
        img_h = torch.flip(batch_img, dims=[-1])
        out2 = model(img_h)

        # vertical flip
        img_v = torch.flip(batch_img, dims=[-2])
        out3 = model(img_v)

        # handle multi-output
        out1 = out1[-1] if isinstance(out1, (list, tuple)) else out1
        out2 = out2[-1] if isinstance(out2, (list, tuple)) else out2
        out3 = out3[-1] if isinstance(out3, (list, tuple)) else out3

        # flip back
        out2 = torch.flip(out2, dims=[-1])
        out3 = torch.flip(out3, dims=[-2])

        # average
        final_output = (out1 + out2 + out3) / 3

        # -------- STRONG TTA END --------

        # -------- THRESHOLDING --------
        probs = torch.softmax(final_output, dim=1)[:, 1, :, :]
        preds = (probs > THRESHOLD).long()
        # --------------------------------

        # ✅ FIXED (important)
        labels_np = labels.cpu().numpy().squeeze()
        preds_np = preds.cpu().numpy().squeeze()

        assd_values.append(compute_assd(preds_np, labels_np))

        tp += np.sum((labels_np == 1) & (preds_np == 1))
        tn += np.sum((labels_np == 0) & (preds_np == 0))
        fp += np.sum((labels_np == 0) & (preds_np == 1))
        fn += np.sum((labels_np == 1) & (preds_np == 0))

# ---------------- Metrics ----------------

eps = 1e-8

precision = tp / (tp + fp + eps)
recall = tp / (tp + fn + eps)
f1 = 2 * precision * recall / (precision + recall + eps)

iou0 = tn / (tn + fp + fn + eps)
iou1 = tp / (tp + fp + fn + eps)
miou = (iou0 + iou1) / 2

oa = (tp + tn) / (tp + tn + fp + fn + eps)

total = tp + tn + fp + fn
po = (tp + tn) / total
pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total * total)

kappa = (po - pe) / (1 - pe + eps)

# ---------------- Output ----------------

print("\nConfusion Matrix")
print(f"TP: {tp}  FP: {fp}")
print(f"FN: {fn}  TN: {tn}")

print("\nFINAL TEST RESULTS")
print(f"Precision:        {round(precision * 100, 2)} %")
print(f"Recall:           {round(recall * 100, 2)} %")
print(f"F1 Score:         {round(f1 * 100, 2)} %")
print(f"Mean IoU:         {round(miou * 100, 2)} %")
print(f"Overall Accuracy: {round(oa * 100, 2)} %")
print(f"Kappa:            {round(kappa * 100, 2)} %")
print(f"ASSD:             {round(mean(assd_values), 4)}")

Overwriting eval.py


In [19]:
!python eval.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Test set size: 186 images
Loading model: runs/train/1/best_checkpoint.pt
100%|█████████████████████████████████████████| 186/186 [00:17<00:00, 10.70it/s]

Confusion Matrix
TP: 780897  FP: 73934
FN: 103275  TN: 11231590

FINAL TEST RESULTS
Precision:        91.35 %
Recall:           88.32 %
F1 Score:         89.81 %
Mean IoU:         89.98 %
Overall Accuracy: 98.55 %
Kappa:            89.03 %
ASSD:             0.8995


In [ ]:
%%writefile eval.py
import torch
from models.DFLNet import DFLNet
from statistics import mean
from tqdm import tqdm
from util.AverageMeter import compute_assd
from util.transforms import test_transforms
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
from glob import glob
from dataset import BreastData
import os
import argparse

# ---------------- Arguments ----------------
parser = argparse.ArgumentParser()
parser.add_argument("--ckpt", type=str, default=None,
                    help="Path to checkpoint (optional)")
args = parser.parse_args()

# ---------------- Device ----------------
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ---------------- Dataset ----------------
test_path = "./datasets/BUSI-WHU/test/"

test_data = pd.DataFrame({
    'images': sorted(glob(test_path + "img/*.bmp")),
    'masks': sorted(glob(test_path + "gt/*.bmp"))
})

if len(test_data) == 0:
    raise FileNotFoundError(f"No test images found at {test_path}")

print(f"Test set size: {len(test_data)} images")

test_dataset = BreastData(test_data, transforms=test_transforms)

test_loader = DataLoader(test_dataset,
                         batch_size=1,
                         shuffle=False,
                         num_workers=2)

# ---------------- Load Model ----------------
if args.ckpt is not None:
    path = args.ckpt
else:
    train_folders = sorted(glob("runs/train/*"))

    if not train_folders:
        raise FileNotFoundError("No training folders found")

    latest_folder = train_folders[-1]
    path = os.path.join(latest_folder, "best_checkpoint.pt")

    if not os.path.exists(path):
        raise FileNotFoundError(f"No best_checkpoint.pt found in {latest_folder}")

print("Loading model:", path)

checkpoint = torch.load(path, map_location=device, weights_only=False)

# Dummy opt
class Opt:
    def __init__(self):
        self.inplanes = 64
        self.network = "DFLNet"
        self.input_size = 256
        self.pretrain_pth = 'None'

opt = Opt()

model = DFLNet(opt).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ---------------- Evaluation ----------------

THRESHOLD = 0.32   # 🔥 BEST RANGE: try 0.30–0.36

tn = fp = fn = tp = 0
assd_values = []

with torch.no_grad():

    for batch_img, labels, _ in tqdm(test_loader):

        batch_img = batch_img.float().to(device)
        labels = labels.long().to(device)

        # -------- TTA (STABLE VERSION) --------

        # original
        out1 = model(batch_img)

        # horizontal flip
        img_h = torch.flip(batch_img, dims=[-1])
        out2 = model(img_h)

        # vertical flip
        img_v = torch.flip(batch_img, dims=[-2])
        out3 = model(img_v)

        # handle outputs
        out1 = out1[-1] if isinstance(out1, (list, tuple)) else out1
        out2 = out2[-1] if isinstance(out2, (list, tuple)) else out2
        out3 = out3[-1] if isinstance(out3, (list, tuple)) else out3

        # flip back
        out2 = torch.flip(out2, dims=[-1])
        out3 = torch.flip(out3, dims=[-2])

        # simple average (BEST STABLE)
        final_output = (out1 + out2 + out3) / 3

        # -------- THRESHOLD --------
        probs = torch.softmax(final_output, dim=1)[:, 1, :, :]
        preds = (probs > THRESHOLD).long()

        # fix shape for ASSD
        labels_np = labels.cpu().numpy().squeeze()
        preds_np = preds.cpu().numpy().squeeze()

        # metrics
        assd_values.append(compute_assd(preds_np, labels_np))

        tp += np.sum((labels_np == 1) & (preds_np == 1))
        tn += np.sum((labels_np == 0) & (preds_np == 0))
        fp += np.sum((labels_np == 0) & (preds_np == 1))
        fn += np.sum((labels_np == 1) & (preds_np == 0))

# ---------------- Metrics ----------------

eps = 1e-8

precision = tp / (tp + fp + eps)
recall = tp / (tp + fn + eps)
f1 = 2 * precision * recall / (precision + recall + eps)

iou0 = tn / (tn + fp + fn + eps)
iou1 = tp / (tp + fp + fn + eps)
miou = (iou0 + iou1) / 2

oa = (tp + tn) / (tp + tn + fp + fn + eps)

total = tp + tn + fp + fn
po = (tp + tn) / total
pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total * total)

kappa = (po - pe) / (1 - pe + eps)

# ---------------- Output ----------------

print("\nConfusion Matrix")
print(f"TP: {tp}  FP: {fp}")
print(f"FN: {fn}  TN: {tn}")

print("\nFINAL TEST RESULTS")
print(f"Precision:        {round(precision * 100, 2)} %")
print(f"Recall:           {round(recall * 100, 2)} %")
print(f"F1 Score:         {round(f1 * 100, 2)} %")
print(f"Mean IoU:         {round(miou * 100, 2)} %")
print(f"Overall Accuracy: {round(oa * 100, 2)} %")
print(f"Kappa:            {round(kappa * 100, 2)} %")
print(f"ASSD:             {round(mean(assd_values), 4)}")

Overwriting eval.py


In [24]:
!python eval.py

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Test set size: 186 images
Loading model: runs/train/1/best_checkpoint.pt
100%|█████████████████████████████████████████| 186/186 [00:17<00:00, 10.80it/s]

Confusion Matrix
TP: 783677  FP: 77090
FN: 100495  TN: 11228434

FINAL TEST RESULTS
Precision:        91.04 %
Recall:           88.63 %
F1 Score:         89.82 %
Mean IoU:         89.98 %
Overall Accuracy: 98.54 %
Kappa:            89.04 %
ASSD:             0.9015
